# Analysis of experiment #5: Repetitions of the pricing heuristic

## Read data

In [1]:
# Read csv

import pandas as pd

df = pd.read_csv("stats5.csv")

# Rename time limit states
df.loc[df.time == 900, "state"] = "TIMELIMIT"
df["timeSolved"] = df.apply(lambda row: float("nan") if row.state == "TIMELIMIT" else row.time, axis=1)

df

,instance,solver,run,nvertices,nedges,n,m,nvars,ncons,state,...,otherNodesNColsMwis1,otherNodesNColsMwis2,otherNodesNColsExact,otherNodesTime,otherNodesTimePool,otherNodesTimeHeur,otherNodesTimeMwis1,otherNodesTimeMwis2,otherNodesTimeExact,timeSolved
0,r_n130_p0.25_nA26_nB13_i1,byp_r1,0,130,2181,26,13,-1,-1,FEASIBLE,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,367.3680
1,r_n170_p0.75_nA34_nB17_i2,byp_r1,0,170,10813,34,17,-1,-1,FEASIBLE,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,385.9420
2,r_n170_p0.75_nA34_nB17_i0,byp_r1,0,170,10814,34,17,-1,-1,TIMELIMIT,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
3,r_n170_p0.75_nA17_nB34_i4,byp_r1,0,170,10756,17,34,-1,-1,OPTIMAL,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,334.5920
4,r_n150_p0.5_nA15_nB30_i4,byp_r1,0,150,5679,15,30,-1,-1,FEASIBLE,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,257.6170
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245,r_n170_p0.75_nA17_nB17_i3,byp_r10000,0,170,10751,17,17,-1,-1,FEASIBLE,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,185.3060
246,r_n150_p0.5_nA15_nB30_i0,byp_r10000,0,150,5548,15,30,-1,-1,OPTIMAL,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,68.7171
247,r_n150_p0.5_nA15_nB15_i2,byp_r10000,0,150,5568,15,15,-1,-1,OPTIMAL,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,140.4660
248,r_n150_p0.5_nA30_nB15_i3,byp_r10000,0,150,5675,30,15,-1,-1,TIMELIMIT,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,NaN


In [2]:
df.columns

Index(['instance', 'solver', 'run', 'nvertices', 'nedges', 'n', 'm', 'nvars',
       'ncons', 'state', 'time', 'nodes', 'nodesLeft', 'lb', 'ub', 'gap',
       'ninfeas', 'ninfeasPrepro', 'ninfeasCheck', 'ninfeasAux', 'nint',
       'ngcp', 'gcpTime', 'nsol', 'nsolHeur', 'nsolLR', 'ntrivial', 'rootlb',
       'rootub', 'rootHeurTime', 'rootFeasTime', 'rootNCalls',
       'rootNCallsPool', 'rootNCallsHeur', 'rootNCallsMwis1',
       'rootNCallsMwis2', 'rootNCallsExact', 'rootNCols', 'rootNColsPool',
       'rootNColsHeur', 'rootNColsMwis1', 'rootNColsMwis2', 'rootNColsExact',
       'rootTime', 'rootTimePool', 'rootTimeHeur', 'rootTimeMwis1',
       'rootTimeMwis2', 'rootTimeExact', 'otherNodesHeurTime',
       'otherNodesFeasNCalls', 'otherNodesFeasTime', 'otherNodesNCalls',
       'otherNodesNCallsPool', 'otherNodesNCallsHeur', 'otherNodesNCallsMwis1',
       'otherNodesNCallsMwis2', 'otherNodesNCallsExact', 'otherNodesNCols',
       'otherNodesNColsPool', 'otherNodesNColsHeur', 'other

## Summary

In [3]:
df3 = df.groupby(["solver"]).agg(
    {
        "state": [("feasible", lambda x: (x == "FEASIBLE").sum()), 
                  ("optimal", lambda x: (x == "OPTIMAL").sum()),
                 ("time limit", lambda x: (x == "TIMELIMIT").sum())],
        "time": [("avg", "mean")],
        "rootTimeHeur": [("avg", "mean")],
        "rootNCallsHeur": [("avg", "mean")],
        "rootNColsHeur": [("avg", "mean")],
        "rootTimeExact": [("avg", "mean")],
        "rootNCallsExact": [("avg", "mean")],
        "rootlb": [("sum", "sum")],
    }
).reset_index()
df3

solver    state                           time rootTimeHeur  \
              feasible optimal time limit         avg          avg   
0      byp_r1       21       6         23  652.700520     0.059459   
1     byp_r10       24       5         21  622.933240     0.193971   
2    byp_r100       28       6         16  529.408240     0.926609   
3   byp_r1000       33       6         11  505.055280     6.801752   
4  byp_r10000       27       6         17  531.520788    59.919543   

  rootNCallsHeur rootNColsHeur rootTimeExact rootNCallsExact     rootlb  
             avg           avg           avg             avg        sum  
0         214.74        124.80    604.129680           89.94  181.70170  
1        1221.40        292.84    574.303268           48.74  181.65388  
2        6964.00        838.82    478.453394           30.68  181.62452  
3       53020.00       4767.80    446.706286           28.26  181.61800  
4      453000.00      39660.28    419.555185           25.90  181.62138